In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_session_status_add;

CREATE TABLE silver_rdm_session_status_add AS

WITH mpb_source AS (
    -- MPB source values derived from DRJ appointments and appointment attendances
    -- session_status_src_id uses same logic as session_status_src_name for MPB001

    SELECT DISTINCT
        CASE
            WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
            ELSE TRIM(att.name)
        END AS session_status_src_name,

        LOWER(TRIM(
            CASE
                WHEN TRIM(a.cancelledBy) IS NOT NULL AND TRIM(a.cancelledBy) <> ''
                    THEN CONCAT(TRIM(att.name), '_', TRIM(a.cancelledBy))
                ELSE TRIM(att.name)
            END
        )) AS session_status_src_id,

        'MPB001' AS session_status_src_sys_inst_id

    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE att.name IS NOT NULL
      AND TRIM(att.name) <> ''
)

SELECT
    s.session_status_src_id,
    s.session_status_src_name,
    s.session_status_src_sys_inst_id
FROM mpb_source s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;

In [ ]:
%%sql
SELECT *
FROM silver_rdm_session_status_add
ORDER BY session_status_src_name;

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows
FROM silver_rdm_session_status_add;

In [ ]:
%%sql
SELECT
    session_status_src_id,
    session_status_src_sys_inst_id,
    COUNT(*) AS cnt
FROM silver_rdm_session_status_add
GROUP BY session_status_src_id, session_status_src_sys_inst_id
HAVING COUNT(*) > 1;